# ಪಾಠ 01 - AI ಏಜೆಂಟ್‌ಗಳಿಗೆ ಪರಿಚಯ

**ಸुरುಮಾಡುವವರಿಗಾಗಿ AI ಏಜೆಂಟ್‌ಗಳು** ಕೋರ್ಸ್‌ನ ಮೊದಲ ಪಾಠಕ್ಕೆ ಸುಸ್ವಾಗತ!

**AI ಏಜೆಂಟ್** ಎಂದರೆ ಒಂದು ಪ್ರೋಗ್ರಾಂ ಆಗಿದ್ದು, ಅದೊಂದು ದೊಡ್ಡ ಭಾಷಾ ಮಾದರಿ (LLM) ಯನ್ನು ಅದರ ತಾರ್ಕಿಕ ಎಂಜಿನ್ ಆಗಿ ಉಪಯೋಗಿಸಿ, *ಕ್ರಿಯೆಗಳು* ಯಥಾರ್ಥ ಜಗತ್ತಿನಲ್ಲಿ ತೆಗೆದುಕೊಳ್ಳಬಹುದು — API ಕರೆ ಮಾಡುವುದು, ಡೇಟಾಬೇಸ್‌ಗಳನ್ನು ವಿಚಾರಿಸುವುದು, ಅಥವಾ ಕೋಡ್ ನಡೆಯಿಸುವುದು — ಬಳಕೆದಾರರನ್ನು ಪರವಾಗಿ ಗುರಿಯನ್ನು ಸಾಧಿಸಲು.

ಈ ನೋಟ್‌ಬುಕ್‌ನಲ್ಲಿ ನೀವು ನಿಮ್ಮ ಮೊದಲ ಏಜೆಂಟ್ ಅನ್ನು ನಿರ್ಮಿಸುವಿರಿ: **ಪ್ರಯಾಣ ಏಜೆಂಟ್** ಇದು ರಜೆ ಕಡೆಗಳನ್ನು ಶಿಫಾರಸು ಮಾಡುತ್ತದೆ. ಇದನ್ನು ಮಾಡುತ್ತಾ ನೀವು ಕಲಿಯುವಿರಿ:

1. **Microsoft Agent Framework** ಬಳಸಿ Azure AI Foundry Agent Service ಗೆ ಸಂಪರ್ಕ ಹೊಂದುವುದು.
2. ಏಜೆಂಟ್‌ಗೆ ಒಂದು **ಸಾಧನ** ಕೊಡುವುದು — ಅದನ್ನು ಕರೆ ಮಾಡಬಹುದಾದ ಸಾಮಾನ್ಯ ಪೈಥಾನ್ ಫಂಕ್ಷನ್.
3. ಏಜೆಂಟ್ ಅನ್ನು ಓಡಿಸಿ ಅದರ ಪ್ರತಿಕ್ರಿಯೆಯನ್ನು ಪರಿಶೀಲಿಸುವುದು.
4. ಏಜೆಂಟ್‌ನ ಪ್ರತಿಕ್ರಿಯೆಯನ್ನು ಟೋಕನ್-ಬೈ-ಟೋಕನ್ ಧಾರಾಕಾರ ತೆಗೆದು ನೋಡುವುದು.


## ಸೆಟಪ್

ಈ ನೋಟ್ಬುಕ್ ಅನ್ನು ಚಾಲನೆಮಾಡುವ ಮುಂಚೆ, ನಿಮಗೆ ಈ ಕೆಳಗಿನವುಗಳಿವೆ ಎಂಬುದು ಖಚಿತಪಡಿಸಿಕೊಳ್ಳಿ:

1. **ನಿಯೋಜಿತ ಚಾಟ್ ಮಾದರಿಯೊಂದಿಗೆ ಒಂದು Azure AI Foundry ಪ್ರಾಜೆಕ್ಟ್** (ಉದಾ: `gpt-4o-mini`).
2. **Azure CLI ಮೂಲಕ ಲಾಗಿನ್ ಆಗಿರುವುದು** — ನಿಮ್ಮ ಟರ್ಮಿನಲ್‌ನಲ್ಲಿ `az login` ಅನ್ನು ಓಡಿಸಿರಿ.
3. **ಅಗತ್ಯವಿರುವ ಪರಿಸರ ಬದಲಾವಣೆಗಳನ್ನು ಹೊಂದಿಸಿ:**
   - `AZURE_AI_PROJECT_ENDPOINT` — ನಿಮ್ಮ Azure AI Foundry ಪ್ರಾಜೆಕ್ಟ್ ಎಂಡ್‌ಪಾಯಿಂಟ್.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — ನಿಮ್ಮ ನಿಯೋಜಿತ ಮಾದರಿಯ ಹೆಸರು.

ಕೆಳಗಿನ ಸೆಲ್ ನಿಮಗೆ ಬೇಕಾಗುವ Python ಪ್ಯಾಕೇಜ್‌ಗಳನ್ನು ಸ್ಥಾಪಿಸುತ್ತದೆ.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## ನಿಮ್ಮ ಮೊದಲ ಏಜೆಂಟ್ ರಚನೆ

ಏಜೆಂಟ್‌ಗೆ ಎರಡು ಅಗತ್ಯಗಳು ಇವೆ:

- ಅದಕ್ಕೆ *ಯಾರು ಇದು* ಮತ್ತು *ಎಂತಹ ವರ್ತನೆ ತೋರಬೇಕು* ಎಂಬುದನ್ನು ಹೇಳುವ **ಸೂಚನೆಗಳು** (ಒಂದು ಸಿಸ್ಟಂ ಪ್ರಾಂಪ್ಟ್).
- ಏಜೆಂಟ್ ಮಾಹಿತಿ ಪಡೆಯಲು ಅಥವಾ ಕ್ರಿಯೆಗಳನ್ನನುಮಿಸಲು ಕರೆಮಾಡಬಹುದಾದ `@tool` ಮೂಲಕ ಅಲಂಕೃತವಾಗಿರುವ Python ಫಂಕ್ಷನ್‌ಗಳಾಗಿ **ಸಾಧನಗಳು**.

ಕೆಳಗೆ ನಾವು ಜನಪ್ರಿಯ ರಜೆ ಸ್ಥಳಗಳ ಪಟ್ಟಿ ನೀಡುವ ಒಂದು ಸರಳ ಸಾಧನವನ್ನು ನಿರ್ಧರಿಸಿದ್ದೇವೆ. ಉಪಯೋಗಿಸಿದರೆ, ಬಳಕೆದಾರರು ಪ್ರಯಾಣ ಶಿಫಾರಸುಗಳನ್ನು ಕೇಳಿದಾಗ ಏಜೆಂಟ್ ಈ ಸಾಧನವನ್ನು ಬಳಸುವುದು.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## ಸ್ಟ್ರೀಮಿಂಗ್ ಪ್ರತಿಕ್ರಿಯೆಗಳು

ಹೆಚ್ಚು ಸಂವಹನಾತ್ಮಕ ಅನುಭವಕ್ಕಾಗಿ ನೀವು ಏಜೆಂಟ್‌ನ ಪ್ರತಿಕ್ರಿಯೆಯನ್ನು **ಸ್ಟ್ರೀಮ್** ಮಾಡಬಹುದು. ಸಂಪೂರ್ಣ ಉತ್ತರದಿಗಾಗಿ ನಿರೀಕ್ಷಿಸುವ ಬದಲು, ಏಜೆಂಟ್ ಉತ್ಪಾದಿಸುವಂತೆ ಪಠ್ಯ ತುಣುಕುಗಳನ್ನು ನೀಡುತ್ತದೆ. ಇದು ನಿಜ ಕಾಲದಲ್ಲಿ ಔಟ್‌ಪುಟ್ ಪ್ರದರ್ಶಿಸಲು ಬಯಸುವ ಚಾಟ್ ಇಂಟರ್ಫೇಸ್‌ಗಳಲ್ಲಿ ವಿಶೇಷವಾಗಿ ಉಪಯುಕ್ತವಾಗಿದೆ.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## ಸಾರಾಂಶ

ಈ ಪಾಠದಲ್ಲಿ ನೀವು ಕೆಳಕಂಡವುಗಳನ್ನು ಕಲಿತುಕೊಂಡಿರಿ:

- **ಪ್ರೊವೈಡರ್ ಸೃಷ್ಟಿಸಿ** ಅದನ್ನು `AzureAIProjectAgentProvider` ಮೂಲಕ Azure AI Foundry Agent ಸೇವೆಗೆ ಸಂಪರ್ಕಿಸುವುದು.
- ಏಜೆಂಟ್ ನಿಮ್ಮ Python ಫಂಕ್ಷನ್‌ಗಳನ್ನು ಕರೆ ಮಾಡುವಂತೆ ಮಾಡಲು `@tool` ಸಲಹೆಗಳೊಂದಿಗೆ **ಒಂದು ಉಪಕರಣವನ್ನು ವ್ಯಾಖ್ಯಾನಿಸಿ**.
- ಬಳಕೆದಾರ ಸಂದೇಶದೊಂದಿಗೆ ಏಜೆಂಟನ್ನು **ನಡೆಗಾಣಿಸಿ** ಮತ್ತು ಅದರ ಪ್ರತಿಕ್ರಿಯೆಯನ್ನು ಮುದ್ರಿಸಿ.
- ನೇರ-ಸಮಯ ಔಟ್‌ಪುಟ್‌ಗಾಗಿ ಉತ್ತರಗಳನ್ನು **ಸ್ರೋತಗೊಳಿಸಿ**.

ಮುಂದಿನ ಪಾಠದಲ್ಲಿ ನಾವು ಏಜೆಂಟಿಕ್ ಫ್ರೇಿಂವರ್ಕ್‌ಗಳನ್ನು ವಿಸ್ತೃತವಾಗಿ ಪರಿಶೀಲಿಸಿ ಏಜೆಂಟ್ಗೆ ಹೆಚ್ಚು ಶಕ್ತिशಾಲಿ ಉಪಕರಣಗಳು ಮತ್ತು ಬಹು-ಹಂತದ ತರ್ಕ ಸಾಮರ್ಥ್ಯಗಳನ್ನು ನೀಡಿ ಕಲಿಯುತ್ತೇವೆ.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ಅಸ್ವೀಕಾರ**:
ಈ ದಸ್ತಾವೇಜು AI ಅನುವಾದ ಸೇವೆ [Co-op Translator](https://github.com/Azure/co-op-translator) ಬಳಸಿ ಅನುವಾದಿಸಲಾಗಿದೆ. ನಾವು ನಿಖರತೆಯನ್ನು ಸಾಧಿಸಲು ಪ್ರಯತ್ನಿಸುತ್ತಿದ್ದರೂ, ದಯವಿಟ್ಟು ಗಮನಿಸಿ, ಸ್ವಯಂಚಾಲಿತ ಅನುವಾದಗಳಲ್ಲಿ ದೋಷಗಳು ಅಥವಾ ಅಸಡ್ಡೆಗಳು ಇರಬಹುದು. ಮೂಲ ಭಾಷೆಯಲ್ಲಿರುವ ಮೂಲ ದಸ್ತಾವೇಜು ಪ್ರಾಮಾಣಿಕ ಮೂಲವೆಂದು ಪರಿಗಣಿಸಬೇಕು. ಪ್ರಮುಖ ಮಾಹಿತಿಗಾಗಿ, ವೃತ್ತಿಪರ ಮಾನವ ಅನುವಾದವನ್ನು ಶಿಫಾರಸು ಮಾಡಲಾಗುತ್ತದೆ. ಈ ಅನುವಾದವನ್ನು ಬಳಸುವ ಮೂಲಕ ಉಂಟಾಗುವ ಯಾವುದೇ ತಪ್ಪು ಅರ್ಥಗಳ ಅಥವಾ ತಪ್ಪು ವ್ಯಾಖ್ಯಾನಗಳ ಬಗ್ಗೆ ನಾವು ಹೊಣೆಗಾರರಲ್ಲ.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
